In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
search_url = os.getenv("SEARCH_SERVICE_URL")
search_api_key = os.getenv("SEARCH_SERVICE_API_KEY")
blob_connection_string = os.getenv("STORAGE_CONNECTION_STRING")
blob_container_name = os.getenv("STORAGE_CONTAINER_NAME")
foundry_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
foundry_model_api_key = os.getenv("FOUNDRY_MODEL_API_KEY")
azure_openai_endpoint = (
    os.getenv("AZURE_OPENAI_ENDPOINT")
    or os.getenv("AZURE_OPENAI_RESOURCE_URL")
    or os.getenv("AZURE_OPENAI_ENDPOINT_URL")
    or "https://foundry-rag-nagh.cognitiveservices.azure.com/"
)
azure_openai_api_key = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("AZURE_OPENAI_KEY")
    or foundry_model_api_key
)
llm_model_name = os.getenv("LLM_MODEL_NAME")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
# print(search_url, search_api_key, blob_connection_string, blob_container_name, azure_openai_endpoint, azure_openai_api_key, llm_model_name, embedding_model_name)

In [3]:
# List knowledge bases by name
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient

index_client = SearchIndexClient(endpoint = search_url, credential = AzureKeyCredential(search_api_key))

for kb in index_client.list_knowledge_bases():
    print(f"  - {kb.name}")

  - health-banking-kb


In [ ]:
# Create a knowledge base using the actual sources already created in the knowledge-source notebook
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import KnowledgeBase, KnowledgeSourceReference
from azure.search.documents.knowledgebases.models import KnowledgeRetrievalMinimalReasoningEffort

if not search_url or not search_api_key:
    raise ValueError("SEARCH_SERVICE_URL and SEARCH_SERVICE_API_KEY must be set in your environment or .env file.")

index_client = SearchIndexClient(
    endpoint=search_url,
    credential=AzureKeyCredential(search_api_key),
)

existing_knowledge_sources = [
    "my-blob-ks-2",
    "ks-fdic-institutions",
    "ks-fdic-locations",
    "ks-fdic-financials",
    "ks-fred-fedfunds",
    "ks-fred-gdp",
    "ks-fred-cpi",
    "ks-fred-unemployment",
    "ks-fred-mortgage",
    "ks-fred-health-spending",
]

knowledge_base = KnowledgeBase(
    name="health-banking-kb",
    description=(
        "Unified knowledge base covering FDIC bank institutions, branch locations, "
        "financial metrics, FRED macro indicators, and the PDF blob source."
    ),
    output_mode="extractiveData",
    knowledge_sources=[
        KnowledgeSourceReference(name=source_name)
        for source_name in existing_knowledge_sources
    ],
    encryption_key=None,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{knowledge_base.name}' created or updated successfully.")

Knowledge base 'health-banking-kb' created or updated successfully.


### Health & Banking Knowledge Base

Creates a single knowledge base that unifies all 13 SQL knowledge sources (FDIC + FRED) and the PDF blob source.

In [ ]:
# Verify all expected knowledge sources exist before creating the KB
existing_ks = {ks.name for ks in index_client.list_knowledge_sources()}
print("Found knowledge sources:")
for name in sorted(existing_ks):
    print(f"  - {name}")

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import KnowledgeRetrievalMinimalReasoningEffort

index_client = SearchIndexClient(
    endpoint=search_url,
    credential=AzureKeyCredential(search_api_key),
)

# Use extractiveData mode — the agent LLM synthesizes the answer, so the KB
# does NOT need to call an LLM endpoint. This avoids the /v1 path api-version error.
knowledge_base = KnowledgeBase(
    name="health-banking-kb",
    description=(
        "Unified knowledge base covering FDIC bank institutions, branch locations, "
        "financial metrics, key FRED macroeconomic indicators (GDP, CPI, fed funds, "
        "mortgage rates, unemployment, health spending), and health/banking PDF documents."
    ),
    output_mode="extractiveData",
    knowledge_sources=[
        KnowledgeSourceReference(name="my-blob-ks-2"),
        KnowledgeSourceReference(name="ks-fdic-institutions"),
        KnowledgeSourceReference(name="ks-fdic-locations"),
        KnowledgeSourceReference(name="ks-fdic-financials"),
        KnowledgeSourceReference(name="ks-fred-fedfunds"),
        KnowledgeSourceReference(name="ks-fred-gdp"),
        KnowledgeSourceReference(name="ks-fred-cpi"),
        KnowledgeSourceReference(name="ks-fred-unemployment"),
        KnowledgeSourceReference(name="ks-fred-mortgage"),
        KnowledgeSourceReference(name="ks-fred-health-spending"),
    ],
    encryption_key=None,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{knowledge_base.name}' updated successfully.")

In [ ]:
# Confirm it was created and list all knowledge bases
for kb in index_client.list_knowledge_bases():
    print(f"  - {kb.name}")

  - health-banking-kb
